In [121]:
from utils.llm_api.client import GPT4AllClient
import os
import re

In [122]:
def clean_python_code(code_string):
    """
    Removes comments, import statements, and print statements from a Python code string.
    """
    # Remove comments (single-line and multi-line)
    code_string = re.sub(r'#.*', '', code_string)  # Single-line comments
    code_string = re.sub(r'""".*?"""|\'\'\'.*?\'\'\'', '', code_string, flags=re.DOTALL)  # Multi-line comments

    # Remove import statements
    code_string = re.sub(r'^\s*import .*|^\s*from .* import .*', '', code_string, flags=re.MULTILINE)

    # Remove empty lines caused by the removals
    code_string = re.sub(r'\n\s*\n', '\n', code_string)

    return code_string.strip()

In [123]:
def clean_true_code_for_comparison(code: str) -> str:
    code_lines =  clean_python_code(code).split('\n')
    
    return '\n'.join(code_lines)

In [124]:
def clean_generated_code_for_comparison(code: str) -> str:
    code_lines = clean_python_code(code).replace("\t", "").split('\n')
    
    return "\n".join(code_lines[1:])

In [139]:
query = """
    You are code quality judge. You have been given two pieces of code. One is the reference code and the other is the generated code. You have to evaluate the generated code based on the reference code. 
    You have to evaluate the generated code based on the following criteria:
    
    1. Correctness: The generated code should produce the same output as the reference code.
    2. Readability: The generated code should be easy to read and understand.
    3. Ignore data loading in the reference code.
    
    The reference code is: 
    %s
    
    The generated code is: 
    %s
    
    Return the quality score as a single float value between 0 and 1, delimited for easy extraction. 
    It is very important to strictly use the format: <<<SCORE: float_value>>>
"""

In [140]:
available_models = GPT4AllClient().get_available_models()
available_models

['Reasoner v1',
 'DeepSeek-R1-Distill-Llama-8B',
 'Llama 3.2 3B Instruct',
 'Llama 3.2 1B Instruct',
 'Llama 3.1 8B Instruct 128k',
 'nomic-ai/nomic-embed-text-v1.5-GGUF']

### Performing evaluation with LLM

In [141]:
MODEL_NAME_JUDGE = available_models[1]
client_judge = GPT4AllClient(model_name=MODEL_NAME_JUDGE)

In [142]:
query_types = ["desc", "geo", "infer"]
reference_dir = "data/correct_results"
generated_dir = "data/generated/%s/%s"
generated_models = os.listdir("data/generated")
generated_models

['DeepSeek-R1-Distill-Llama-8B',
 'Llama 3.1 8B Instruct 128k',
 'Llama 3.2 3B Instruct',
 'Reasoner v1']

In [143]:
llm_eval_dir = "data/llm_evaluation"
os.makedirs(llm_eval_dir, exist_ok=True)

In [144]:
model = generated_models[1]
query_type = 'geo'

In [145]:
generated_folder = generated_dir % (model, query_type)
generated_files = os.listdir(generated_folder)
reference_folder = os.path.join(reference_dir, query_type)
reference_files = os.listdir(reference_folder)

matching_files = sorted(set(generated_files).intersection(reference_files))
results_folder = os.path.join(llm_eval_dir, model, query_type)
os.makedirs(results_folder, exist_ok=True)
for file in matching_files:
    with open(os.path.join(generated_folder, file), "r") as f:
        generated_code = f.read()
    with open(os.path.join(reference_folder, file), "r") as f:
        reference_code = f.read()
    
    reference_code = clean_true_code_for_comparison(reference_code)
    generated_code = clean_generated_code_for_comparison(generated_code)
    query_text = query % (reference_code, generated_code)
    response = client_judge.query(query_text, temperature=0)            
    with open(os.path.join(results_folder, file), "w") as f:
        f.write(response)

### Results Overview

In [147]:
evaluated_models = os.listdir(llm_eval_dir)

In [148]:
def extract_score(text):
    match = re.search(r'<<<SCORE:\s*([\d\.]+)>>>', text)
    return float(match.group(1)) if match else None

In [149]:
for evaluated_model in evaluated_models:
    evaluated_query_types = os.listdir(os.path.join(llm_eval_dir, evaluated_model))
    for query_type in evaluated_query_types:
        results_folder = os.path.join(llm_eval_dir, evaluated_model, query_type)
        results_files = os.listdir(results_folder)
        scores = []
        for file in results_files:
            with open(os.path.join(results_folder, file), "r") as f:
                score = extract_score(f.read())
                if score is not None and score >= 0 and score <= 1:
                    scores.append(score)
        print(f"Model: {evaluated_model}, Query Type: {query_type}, Average Score: {sum(scores) / len(scores):.3f}, Valid Scores: {scores}")

Model: Llama 3.1 8B Instruct 128k, Query Type: geo, Average Score: 0.713, Valid Scores: [0.7, 0.8, 0.7, 0.85, 0.85, 0.5, 0.7, 0.6]
